[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-08.ipynb)

# 8주차 실습: 중간고사 대비 자가 점검과 PEFT 심화

**목표.** 1~7주차의 핵심 계산을 **먼저 손으로 예측하고, 셀을 실행해 확인**한다. 새 주제는 없다.

- 문항마다 문제를 읽고, 종이나 아래 답 칸에 예측을 먼저 적는다
- 그다음 셀을 실행해 예측과 비교한다. 틀렸으면 문항 끝에 적힌 주차의 강의 노트로 돌아간다
- 3주차 PEFT 문항은 실습 모델 Qwen3-0.6B-Base에 LoRA, DoRA를 실제로 붙여 학습 파라미터 수를 세어 본다(PEFT 심화)

이 노트북에는 바꿀 곳(TODO)이 없다. 처음부터 끝까지 그대로 실행하면 된다. 문항은 시험 유형을 연습하려고 만든 예시이며 실제 시험 문항이 아니다.

작성 범위와 예측 기록:

- 아래 `실행 전 계산 예상`은 노트북 실행 전에 문제, 제공 공식, 강의 노트를 참고해 작성한 보조 에이전트의 계산. 사용자의 독립적인 손풀이 또는 실제 오답 경험을 대신 기록한 것이 아님.
- 사용자가 확인이 필요하다고 지정한 개념은 **LoRA와 QLoRA의 메모리 계산**. 학습자의 직접 풀이와 설명을 관찰한 기록은 없으며, 실행 결과와 개념 설명을 구분하여 기록.
- 원본의 23개 셀과 계산 순서는 유지. 설치 셀은 없는 패키지만 지정 버전으로 설치하고 기존 환경의 버전을 기록하도록 보완하며, 모델 설정의 revision 고정 및 CPU 실행 환경 기록만 보완.


## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

In [1]:
# 환경 보완: 없는 패키지만 지정 버전으로 설치하고 기존 패키지는 유지한다.
import importlib.metadata
import subprocess
import sys

required = {"torch": "2.8.0", "transformers": "4.57.6", "peft": "0.18.1"}
missing = []
for package, expected in required.items():
    try:
        importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        missing.append(f"{package}=={expected}")
if missing:
    print("없는 패키지만 설치:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *missing])
    importlib.invalidate_caches()

for package, expected in required.items():
    actual = importlib.metadata.version(package)
    print(f"{package}: {actual} (기준 버전 {expected})")
    if actual.split("+")[0] != expected:
        print("  버전 차이 있음: 결과 비교 시 차이를 기록할 것")

import torch
torch.set_num_threads(4)
print("Python:", sys.version.split()[0])
print("실행 장치: CPU / CPU threads:", torch.get_num_threads())
print("PEFT 점검: 무작위 모델의 파라미터 개수 확인, 모델 학습 없음")


torch: 2.8.0+cpu (기준 버전 2.8.0)
transformers: 4.57.6 (기준 버전 4.57.6)
peft: 0.18.1 (기준 버전 0.18.1)


Python: 3.10.0
실행 장치: CPU / CPU threads: 4
PEFT 점검: 무작위 모델의 파라미터 개수 확인, 모델 학습 없음


In [2]:
import math, torch
torch.manual_seed(0)
def check(name, got, expected, tol=1e-3):
    ok = abs(got - expected) <= tol * max(1, abs(expected))
    print(f"[{'맞음' if ok else '다름'}] {name}: 계산 {got:,.4f} / 기대 {expected:,.4f}")

## 1. 어텐션 한 칸 손계산 (1주차)

토큰 3개, 차원 2인 Query, Key, Value가 있다. 첫 토큰의 Query로 어텐션을 계산한다.

- q = (1, 0), K = [(1, 0), (0, 1), (1, 1)], V = [(1, 0), (0, 1), (1, 1)]
- 점수 = q · k / √2 (차원 2의 제곱근으로 나눈다)

**예측해 보기.** (1) 세 점수는? (2) 소프트맥스 뒤 첫 칸의 가중치는? (3) 출력 벡터는? (4) 토큰이 1,000개면 점수표는 몇 칸인가?

**실행 전 계산 예상 (작성 보조)**

- 점수는 `(1/sqrt(2), 0, 1/sqrt(2))`, 약 `(0.7071, 0, 0.7071)`.
- 첫 가중치는 `exp(1/sqrt(2)) / (2*exp(1/sqrt(2)) + 1)`, 약 `0.4011`. 전체 가중치는 약 `(0.4011, 0.1978, 0.4011)`.
- 출력은 `(w1+w3, w2+w3)`, 약 `(0.8022, 0.5989)`.
- 토큰 1,000개의 전체 점수표는 `1,000 * 1,000 = 1,000,000`칸.


In [3]:
q = torch.tensor([1.0, 0.0])
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
V = K.clone()
scores = K @ q / math.sqrt(2)
w = torch.softmax(scores, dim=0)
out = w @ V
print("점수:", [round(x, 4) for x in scores.tolist()])
print("가중치:", [round(x, 4) for x in w.tolist()])
print("출력:", [round(x, 4) for x in out.tolist()])
check("첫 칸 가중치", w[0].item(), math.exp(1 / math.sqrt(2)) / (2 * math.exp(1 / math.sqrt(2)) + 1))
print(f"토큰 1,000개의 점수표: {1000 * 1000:,}칸 (n x n)")

점수: [0.7071, 0.0, 0.7071]
가중치: [0.4011, 0.1978, 0.4011]
출력: [0.8022, 0.5989]
[맞음] 첫 칸 가중치: 계산 0.4011 / 기대 0.4011
토큰 1,000개의 점수표: 1,000,000칸 (n x n)


## 2. 학습 루프 한 걸음 (2주차)

모델 ŷ = w · x, 손실 L = (ŷ - y)². x = 2, y = 10, 처음 w = 3, 학습률 0.1이다.

**예측해 보기.** (1) 처음 손실은? (2) 기울기 dL/dw는? (3) 한 걸음 뒤 w와 손실은?

**실행 전 계산 예상 (작성 보조)**

- 처음 예측값은 `3*2 = 6`, 손실은 `(6-10)^2 = 16`.
- 기울기는 `2*(3*2-10)*2 = -16`.
- 갱신 후 가중치는 `3-0.1*(-16) = 4.6`, 새 손실은 `(4.6*2-10)^2 = 0.64`. 손실 감소 예상.


In [4]:
w = torch.tensor(3.0, requires_grad=True)
x, y, lr = 2.0, 10.0, 0.1
loss = (w * x - y) ** 2
loss.backward()
print(f"처음 손실 {loss.item():.2f}, 기울기 {w.grad.item():.2f}")
with torch.no_grad():
    w_new = w - lr * w.grad
new_loss = (w_new * x - y) ** 2
print(f"한 걸음 뒤 w = {w_new.item():.2f}, 손실 {new_loss.item():.2f}")
check("기울기 2 x (wx - y) x x", w.grad.item(), 2 * (3 * 2 - 10) * 2)

처음 손실 16.00, 기울기 -16.00
한 걸음 뒤 w = 4.60, 손실 0.64
[맞음] 기울기 2 x (wx - y) x x: 계산 -16.0000 / 기대 -16.0000


## 3. PEFT 심화 - Qwen3-0.6B에 LoRA를 붙이면 (3주차)

3주차는 KoELECTRA(1.1억 개)에 LoRA를 붙였다. 같은 계산을 4~7주차 실습 모델 Qwen3-0.6B-Base에 적용한다. 설정만 내려받아 무작위 가중치로 모델을 만든다(가중치 파일은 내려받지 않는다). 파라미터 **개수**를 세는 데는 가중치 값이 필요 없기 때문이다.

한 층의 선형 층 모양(입력 -> 출력)은 다음과 같다. 이 층이 28번 반복된다.

| 모듈 | 모양 | 모듈 | 모양 |
|---|---|---|---|
| q_proj | 1024 -> 2048 | gate_proj | 1024 -> 3072 |
| k_proj | 1024 -> 1024 | up_proj | 1024 -> 3072 |
| v_proj | 1024 -> 1024 | down_proj | 3072 -> 1024 |
| o_proj | 2048 -> 1024 | | |

**예측해 보기.** 랭크 r의 LoRA는 입력 a, 출력 b인 층에 r x (a + b)개를 붙인다.
(1) r = 8로 q_proj, v_proj에만 붙이면 28층 전체 몇 개인가? (2) 일곱 모듈 모두에 붙이면? (3) DoRA는 붙인 층마다 크기 벡터(출력 차원 길이)를 더한다. q_proj, v_proj에 r = 8 DoRA면 LoRA보다 몇 개 많은가?

**실행 전 계산 예상 (작성 보조)**

- q_proj와 v_proj: `28*8*((1024+2048)+(1024+1024)) = 1,146,880`개.
- 일곱 모듈 전체: `28*8*(3072+2048+2048+3072+4096+4096+4096) = 5,046,272`개. q_proj와 v_proj만 붙인 경우의 `4.4`배.
- DoRA의 추가 크기 벡터: `28*(2048+1024) = 86,016`개. 따라서 q_proj와 v_proj의 r=8 DoRA는 `1,232,896`개.
- 파라미터 값은 무작위여도 행렬 모양과 대상 모듈이 같으면 개수는 동일하다는 예상. 실제 PEFT 모듈 부착 후 아래 출력과 비교.


In [5]:
from transformers import AutoConfig, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# 재현성 보완: 설정 파일만 내려받으며 모델 가중치는 내려받지 않는다.
model_revision = "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"
cfg = AutoConfig.from_pretrained("Qwen/Qwen3-0.6B-Base", revision=model_revision)
print("설정 revision:", getattr(cfg, "_commit_hash", model_revision))

def count(targets, r=8, dora=False):
    base = AutoModelForCausalLM.from_config(cfg, dtype=torch.bfloat16)  # 무작위 가중치, 개수만 센다
    total = sum(p.numel() for p in base.parameters())
    pm = get_peft_model(base, LoraConfig(r=r, lora_alpha=2 * r, target_modules=targets, use_dora=dora))
    trainable = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    return trainable, total

L = cfg.num_hidden_layers
shape = {"q_proj": (1024, 2048), "k_proj": (1024, 1024), "v_proj": (1024, 1024), "o_proj": (2048, 1024),
         "gate_proj": (1024, 3072), "up_proj": (1024, 3072), "down_proj": (3072, 1024)}
def lora_formula(targets, r=8):
    return L * sum(r * (shape[t][0] + shape[t][1]) for t in targets)

qv = ["q_proj", "v_proj"]
all7 = list(shape)
t_qv, total = count(qv)
t_all, _ = count(all7)
t_dora, _ = count(qv, dora=True)
print(f"모델 전체 파라미터: {total:,}")
print(f"LoRA r=8, q·v:      {t_qv:,} ({t_qv / total:.3%})  공식 {lora_formula(qv):,}")
print(f"LoRA r=8, 일곱 모듈: {t_all:,} ({t_all / total:.3%})  공식 {lora_formula(all7):,}")
print(f"DoRA r=8, q·v:      {t_dora:,}  LoRA보다 {t_dora - t_qv:,}개 많다 (크기 벡터 (2048 + 1024) x {L}층 = {(2048 + 1024) * L:,})")
check("LoRA q·v 공식과 실제", t_qv, lora_formula(qv), tol=0)

C:\Users\User\Desktop\School2\tmp\deepnlp_20261007\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


설정 revision: da87bfb608c14b7cf20ba1ce41287e8de496c0cd


모델 전체 파라미터: 596,049,920
LoRA r=8, q·v:      1,146,880 (0.192%)  공식 1,146,880
LoRA r=8, 일곱 모듈: 5,046,272 (0.847%)  공식 5,046,272
DoRA r=8, q·v:      1,232,896  LoRA보다 86,016개 많다 (크기 벡터 (2048 + 1024) x 28층 = 86,016)
[맞음] LoRA q·v 공식과 실제: 계산 1,146,880.0000 / 기대 1,146,880.0000


### 3-1. 학습 메모리 어림 - 3주차의 16바이트 규칙

3주차 규칙: 학습하는 파라미터는 가중치 4 + 그래디언트 4 + Adam 기록 8 = **16바이트**, 얼린 파라미터는 가중치만 4바이트(32비트)다. QLoRA는 얼린 선형 층을 4비트(0.5바이트)로 올린다.

여기서 하나 더 본다. QLoRA가 4비트로 바꾸는 것은 보통 **선형 층**이다. 이 모델은 어휘가 151,936개라 임베딩이 크고, 임베딩은 4비트로 바꾸지 않는 경우가 많다. 아래는 임베딩을 16비트(2바이트)로 두는 경우로 어림한다. 활성값과 양자화 보조 숫자는 빠진 어림이다.

**예측해 보기.** 전체 미세조정, LoRA(q·v, r=8), QLoRA(같은 어댑터) 중 어느 것이 몇 GiB쯤일까?

**실행 전 계산 예상 (작성 보조)**

- 강의 노트의 기반 모델 총수 `596,049,920`개, 임베딩 `151,936*1,024 = 155,582,464`개를 사용한 예상. 선형 층 합계 `440,401,920`개, 나머지 `65,536`개.
- 전체 미세조정: `596,049,920*16 / 2^30`, 약 `8.88 GiB`.
- LoRA: `(596,049,920*4 + 1,146,880*16) / 2^30`, 약 `2.24 GiB`.
- QLoRA: `(440,401,920*0.5 + (155,582,464+65,536)*2 + 1,146,880*16) / 2^30`, 약 `0.51 GiB`.
- QLoRA 어림 중 임베딩 몫은 `155,582,464*2 / 549,847,040`, 약 `57%`. 실제 메모리 측정이 아니라 주어진 바이트 규칙에 따른 계산 예상.


In [6]:
emb = cfg.vocab_size * cfg.hidden_size              # 입력 임베딩 (출력 층과 공유)
linear = L * sum(a * b for a, b in shape.values())  # 28층의 선형 층 가중치
rest = total - emb - linear                         # 정규화 층 등
GiB = 2 ** 30
full = total * 16
lora = total * 4 + t_qv * 16
qlora = linear * 0.5 + (emb + rest) * 2 + t_qv * 16
print(f"임베딩 {emb:,} ({emb / total:.1%}), 선형 층 {linear:,} ({linear / total:.1%}), 나머지 {rest:,}")
print(f"전체 미세조정  {full / GiB:5.2f} GiB  (모두 16바이트)")
print(f"LoRA          {lora / GiB:5.2f} GiB  (얼린 가중치 4바이트 + 어댑터 16바이트)")
print(f"QLoRA         {qlora / GiB:5.2f} GiB  (선형 층 0.5바이트, 임베딩 등 2바이트, 어댑터 16바이트)")
print(f"QLoRA에서 임베딩이 차지하는 몫: {(emb * 2) / qlora:.0%}")

임베딩 155,582,464 (26.1%), 선형 층 440,401,920 (73.9%), 나머지 65,536
전체 미세조정   8.88 GiB  (모두 16바이트)
LoRA           2.24 GiB  (얼린 가중치 4바이트 + 어댑터 16바이트)
QLoRA          0.51 GiB  (선형 층 0.5바이트, 임베딩 등 2바이트, 어댑터 16바이트)
QLoRA에서 임베딩이 차지하는 몫: 57%


## 4. 라벨 확률 (4주차)

프롬프트 다음 토큰 자리에서 세 라벨 첫 토큰의 로짓이 주차 2.0, 시설 1.0, 요금 0.0이다.

**예측해 보기.** (1) 소프트맥스 뒤 세 확률은? (2) 주차와 시설의 확률 비는? (3) 로짓에 모두 5를 더하면 확률은 어떻게 되는가?

**실행 전 계산 예상 (작성 보조)**

- 확률은 `(e^2, e, 1)/(e^2+e+1)`, 약 `(0.6652, 0.2447, 0.0900)`.
- 주차/시설 확률 비는 `e^(2-1) = e`, 약 `2.7183`.
- 모든 로짓에 5를 더하면 분자와 분모의 공통 인자 `e^5`가 약분되므로 세 확률 유지.


In [7]:
logits = torch.tensor([2.0, 1.0, 0.0])
p = torch.softmax(logits, dim=0)
print("확률 (주차, 시설, 요금):", [round(x, 4) for x in p.tolist()])
print(f"주차 / 시설 = {p[0] / p[1]:.4f} (e^1 = {math.e:.4f})")
print("로짓 + 5:", [round(x, 4) for x in torch.softmax(logits + 5, dim=0).tolist()], "-> 그대로 (차이만 중요하다)")

확률 (주차, 시설, 요금): [0.6652, 0.2447, 0.09]
주차 / 시설 = 2.7183 (e^1 = 2.7183)
로짓 + 5: [0.6652, 0.2447, 0.09] -> 그대로 (차이만 중요하다)


## 5. 지표와 pass@k (5주차)

긴급 문의 분류 결과: 실제 긴급 10건 중 8건을 긴급으로 잡았고, 실제 일반 30건 중 6건을 긴급으로 잘못 올렸다.

**예측해 보기.** (1) 긴급 부류의 정밀도, 재현율, F1은? (2) 정확도는?

같은 문제를 n = 10번 풀어 c = 3번 맞혔다. **(3)** pass@1과 pass@5의 추정값은? 추정식은 1 - C(n - c, k) / C(n, k)다.

**실행 전 계산 예상 (작성 보조)**

- `TP=8, FN=2, FP=6, TN=24`. 정밀도 `8/14 = 0.5714`, 재현율 `8/10 = 0.8000`, F1 `16/24 = 0.6667`, 정확도 `32/40 = 0.8000`.
- pass@1은 `3/10 = 0.3000`.
- pass@5는 `1-C(7,5)/C(10,5) = 1-21/252 = 11/12`, 약 `0.9167`. 1회 성공 비율과 여러 시도 중 한 번 이상 성공할 비율의 구별.


In [8]:
tp, fn, fp, tn = 8, 2, 6, 24
precision = tp / (tp + fp); recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
acc = (tp + tn) / (tp + fn + fp + tn)
print(f"정밀도 {precision:.4f}, 재현율 {recall:.4f}, F1 {f1:.4f}, 정확도 {acc:.4f}")

def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1 - math.comb(n - c, k) / math.comb(n, k)
for k in [1, 5]:
    print(f"pass@{k} = {pass_at_k(10, 3, k):.4f}")
check("pass@1 = c / n", pass_at_k(10, 3, 1), 0.3)

정밀도 0.5714, 재현율 0.8000, F1 0.6667, 정확도 0.8000
pass@1 = 0.3000
pass@5 = 0.9167
[맞음] pass@1 = c / n: 계산 0.3000 / 기대 0.3000


## 6. 코사인 유사도와 시각 토큰 (6주차)

**예측해 보기.** (1) (3, 4)와 (4, 3)의 코사인 유사도는? (2) 두 라벨의 코사인이 0.266, 0.274이고 logit_scale이 100이면 두 라벨의 확률 비는? (3) SmolVLM에서 640x640 사진은 분할을 켜면 몇 토큰인가? (긴 변을 2048로 키우고 512 타일로 자른 뒤 전체 사진 1장을 더한다. 장당 64토큰)

**실행 전 계산 예상 (작성 보조)**

- 코사인 유사도는 `(3*4+4*3)/(5*5) = 24/25 = 0.96`.
- 더 높은 코사인 라벨(0.274)의 더 낮은 라벨(0.266)에 대한 확률 비는 `exp(100*(0.274-0.266)) = exp(0.8)`, 약 `2.2255`.
- 주어진 분할 조건에서 `4*4+1 = 17`장, 장당 64토큰이므로 `1,088`토큰.


In [9]:
a, b = torch.tensor([3.0, 4.0]), torch.tensor([4.0, 3.0])
cos = torch.nn.functional.cosine_similarity(a, b, dim=0).item()
check("cos((3,4),(4,3))", cos, 0.96)
ratio = math.exp(100 * (0.274 - 0.266))
print(f"확률 비 e^(100 x 0.008) = {ratio:.4f}")
tiles = (2048 // 512) * (2048 // 512) + 1
print(f"640x640 -> 2048x2048, 타일 {tiles - 1}장 + 전체 1장 = {tiles}장 x 64 = {tiles * 64:,}토큰")

[맞음] cos((3,4),(4,3)): 계산 0.9600 / 기대 0.9600
확률 비 e^(100 x 0.008) = 2.2255
640x640 -> 2048x2048, 타일 16장 + 전체 1장 = 17장 x 64 = 1,088토큰


## 7. KV 캐시와 RoPE (7주차)

설정: 층 28, KV 헤드 8, 헤드 차원 128, bf16(2바이트), RoPE 기저 1,000,000, 최대 위치 32,768.

**예측해 보기.** (1) 토큰당 KV 캐시는 몇 바이트인가? 8,192토큰이면? (2) KV 헤드가 16개라면 몇 배인가? (3) 가장 빠른 차원 쌍(k = 1)의 주기는? (4) 위치 보간으로 131,072까지 늘리면 위치 65,536은 몇이 되는가?

**실행 전 계산 예상 (작성 보조)**

- 토큰당 KV 캐시는 `2*28*8*128*2 = 114,688`바이트, 즉 `112 KiB`. 앞의 2는 K와 V, 뒤의 2는 bf16 바이트 수.
- 8,192토큰에서는 `114,688*8,192 = 939,524,096`바이트, 즉 `896 MiB`.
- KV 헤드만 8개에서 16개로 바꾸면 캐시는 2배.
- 가장 빠른 차원 쌍(k=1)의 주기는 `2*pi*b^0 = 2*pi`, 약 `6.28`토큰.
- 위치 보간 결과는 `65,536*32,768/131,072 = 16,384`. 최대 위치를 4배로 늘리는 설정에서 원래 위치 좌표를 4로 나누는 계산.


In [10]:
per_tok = 2 * 28 * 8 * 128 * 2
print(f"토큰당 {per_tok:,}바이트 = {per_tok / 1024:.0f} KiB, 8,192토큰 {per_tok * 8192 / 2**20:,.0f} MiB")
print(f"KV 헤드 16개면 {2 * 28 * 16 * 128 * 2 / per_tok:.0f}배")
b, d = 1_000_000, 128
print(f"k = 1 주기: 2π x b^0 = {2 * math.pi:.2f}토큰, k = 64 주기: {2 * math.pi * b ** ((d - 2) / d):,.0f}토큰")
print(f"위치 보간: 65,536 x 32,768 / 131,072 = {65536 * 32768 // 131072:,}")

토큰당 114,688바이트 = 112 KiB, 8,192토큰 896 MiB
KV 헤드 16개면 2배
k = 1 주기: 2π x b^0 = 6.28토큰, k = 64 주기: 5,063,256토큰
위치 보간: 65,536 x 32,768 / 131,072 = 16,384


## 8. 돌아보기

1. 틀린 문항은 몇 번이었나요? 각 문항이 어느 주차의 어떤 개념에서 왔는지 적어 보세요.
2. 3에서 LoRA를 일곱 모듈 모두에 붙이면 q·v에만 붙일 때보다 몇 배가 되었나요? 그래도 전체의 몇 %인가요?
3. 3-1에서 QLoRA의 메모리 중 임베딩이 차지하는 몫은 얼마였나요? 어휘가 큰 모델에서 QLoRA의 이득이 3주차 표보다 작아지는 이유를 한 문장으로 써 보세요.
4. 1~7주차 개념 위계에서 세 문장으로 설명하지 못한 마디는 어디였나요?

답은 아래 셀에 글로 적으면 됩니다.

1. **개인 오답 수와 계산 검증 결과의 구분.** 사용자의 독립적인 예측 답안이 제공되지 않았으므로, 사용자가 틀린 문항 수나 오답 경험을 만들어 기록하지 않음. 대신 실행 전에 문제와 강의 자료를 참고해 작성한 보조 에이전트의 계산 예상은 1~7번 및 3-1 보충 계산에서 표시 정밀도 기준으로 실행 결과와 모두 일치하여, 이 비교의 계산 불일치는 0건. 이것이 사용자의 개인 정답률이나 개념 숙련도 확인을 뜻하지는 않음.

   | 문항 | 해당 주차와 개념 | 실행 결과로 확인한 주요 값 |
   |---|---|---|
   | 1 | 1주차, 스케일 점수와 소프트맥스 가중합 | 첫 가중치 0.4011, 출력 (0.8022, 0.5989), 점수표 1,000,000칸 |
   | 2 | 2주차, 손실과 기울기 및 갱신 | 처음 손실 16, 기울기 -16, 갱신 후 w=4.6와 손실 0.64 |
   | 3 | 3주차, LoRA의 저랭크 행렬과 DoRA 크기 벡터 | q_proj/v_proj LoRA 1,146,880개, 일곱 모듈 LoRA 5,046,272개, DoRA 추가 86,016개 |
   | 3-1 | 3주차, 동결 가중치와 학습 상태 메모리 | 전체/LoRA/QLoRA 어림 8.88/2.24/0.51 GiB |
   | 4 | 4주차, 로짓과 라벨 확률 | 확률 (0.6652, 0.2447, 0.0900), 확률 비 2.7183, 공통 로짓 이동 시 동일 |
   | 5 | 5주차, 정밀도/재현율/F1와 pass@k | 0.5714/0.8000/0.6667, 정확도 0.8000, pass@1 0.3000, pass@5 0.9167 |
   | 6 | 6주차, 코사인과 시각 토큰 | 코사인 0.96, 높은 라벨/낮은 라벨 확률 비 2.2255, 1,088토큰 |
   | 7 | 7주차, KV 캐시와 RoPE 위치 | 112 KiB/토큰, 8,192토큰에서 896 MiB, KV 헤드 2배면 캐시 2배, 빠른 주기 6.28토큰, 보간 위치 16,384 |

2. **일곱 모듈 LoRA는 q_proj와 v_proj만 붙인 LoRA의 4.4배.** 실제 부착 뒤 센 값으로 `5,046,272 / 1,146,880 = 4.4`이며, 기반 모델 전체 `596,049,920`개 대비 `5,046,272 / 596,049,920 * 100`, 약 `0.847%`에 해당. 파라미터 수 공식과 실제 부착 수의 일치 확인.

   | 설정 | 실제 학습 가능 파라미터 수 | 기반 모델 전체 대비 |
   |---|---:|---:|
   | LoRA r=8, q_proj와 v_proj | 1,146,880 | 약 0.192% |
   | LoRA r=8, 일곱 모듈 | 5,046,272 | 약 0.847% |
   | DoRA r=8, q_proj와 v_proj | 1,232,896 | 약 0.207% |

   DoRA의 추가분은 출력 차원별 크기 벡터 `(2,048+1,024)*28 = 86,016`개이며, 이 셀에서 성능 개선이나 학습 속도를 측정한 것은 아님. 무작위 기반 모델에 실제 PEFT 어댑터를 붙여 개수만 확인한 결과.

3. **QLoRA 어림에서 임베딩이 차지하는 몫은 약 57%.** 임베딩은 `155,582,464*2 = 311,164,928`바이트이고, 전체 어림은 `440,401,920*0.5 + (155,582,464+65,536)*2 + 1,146,880*16 = 549,847,040`바이트이므로, 임베딩 몫은 약 `56.59%`이며 출력에서 정수 비율 `57%`로 반올림됨. 어휘가 큰 이 모델에서는 4비트로 줄이지 않는 큰 임베딩을 16비트로 유지하므로, 선형 가중치를 모두 0.5바이트로 계산하는 단순한 어림보다 QLoRA의 절약 폭이 작아짐.

4. **사용자가 점검이 필요하다고 지정한 마디는 적응(3주차)의 LoRA와 QLoRA 메모리 계산.** 사용자가 실제로 세 문장 설명에 실패했는지는 관찰하지 않았으며, 아래는 확인을 돕기 위해 정리한 세 문장 설명.

   LoRA는 기반 가중치를 얼려 그 부분의 그래디언트와 Adam 기록을 줄이지만 가중치 자체의 저장 공간은 남으므로, 학습 파라미터가 약 0.192%여도 메모리 어림은 전체 미세조정 8.88 GiB의 약 4분의 1인 2.24 GiB 규모임. QLoRA는 동결 선형 가중치를 4비트로 줄이면서 어댑터를 학습하는 방식이며, 이 예에서는 임베딩과 정규화 층을 16비트로 유지하고 학습 어댑터는 16바이트 규칙으로 계산함. 따라서 선형 층 0.5바이트, 임베딩과 나머지 층 2바이트, 어댑터 16바이트를 각각 곱해 더해야 하며, 합계 약 0.51 GiB 중 큰 임베딩이 약 57%를 차지하므로 어댑터 비율만으로 전체 메모리를 판단할 수 없음.

   추가 확인 기준은 `동결 파라미터도 메모리에 남는가`, `어느 모듈만 4비트가 되는가`, `바이트를 GiB로 나눌 때 2^30을 쓰는가`의 세 구분. 실제 학습자 이해도는 이 세 구분을 직접 설명한 응답이 없으므로 판정하지 않음.

검증 범위:

- 원본 23개 셀 유지, 10개 코드 셀을 순서대로 실제 실행. 보조 에이전트의 실행 전 계산 예상과 실행 출력 분리 기록.
- 공개 설정 revision `da87bfb608c14b7cf20ba1ce41287e8de496c0cd`로 만든 무작위 Qwen 모델에 실제 LoRA와 DoRA를 부착하여 파라미터 수 확인. 사전학습 가중치 다운로드, Qwen 학습, 성능 평가는 없음. 2번의 스칼라 손실과 한 걸음 갱신 예제는 원본대로 실행.
- 3-1 메모리는 주어진 바이트 규칙으로 계산한 어림이며 프로세스 RAM이나 GPU 메모리 실측이 아님. 실제 계수 세기는 bfloat16 무작위 모델에서 수행했지만, 메모리 비교는 문제에서 정한 전체 미세조정/LoRA의 32비트 규칙과 QLoRA의 혼합 저장 규칙을 각각 적용.
- 활성값, 양자화 보조 정보, 작업 버퍼와 프레임워크 오버헤드는 계산에서 제외. 따라서 이 숫자가 특정 장치에서 학습이 가능함을 보증하는 것은 아님.

- 위 계산 출력은 전체 순차 실행에서 얻은 결과를 유지. 이후 새 환경의 실행 편의를 위해 준비 셀에 없는 패키지만 설치하는 처리를 추가하고, 준비 셀만 새 커널에서 다시 실행하여 확인. 기존 환경에 모든 패키지가 있어 추가 설치는 없었으며, 계산 셀은 다시 실행하지 않음.


## 9. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-08/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.